# Mobility and policy raw data visualisation

Each series is min–max scaled within the plotted window so co-movement is easy to see despite different native units (mobility multipliers near 1 vs OxCGRT restriction intensity on $[0,1]$).

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib.lines import Line2D

from emu_renewal.constants import MOBILITY_SMOOTH_PERIOD, SOURCE_COLOURS
from emu_renewal.inputs import (
    get_fb_singletile_mobility,
    get_fb_visited_mobility,
    get_google_mobility,
    get_oxcgrt,
)

plt.style.use("ggplot")

In [ ]:
iso3 = "GBR"
start = pd.Timestamp("2020-03-01")
end = pd.Timestamp("2021-06-30")
smooth = MOBILITY_SMOOTH_PERIOD

google_colour = SOURCE_COLOURS["g_mob"]
facebook_colour = SOURCE_COLOURS["fb_visited_mob"]
oxcgrt_colour = SOURCE_COLOURS["oxcgrt_floored"]

In [ ]:
def clip_window(df: pd.DataFrame | pd.Series) -> pd.DataFrame | pd.Series:
    out = df.loc[(df.index >= start) & (df.index <= end)]
    return out.dropna(how="all") if isinstance(out, pd.DataFrame) else out.dropna()


def minmax_within_window(series: pd.Series) -> pd.Series:
    lo, hi = series.min(), series.max()
    if hi > lo:
        return (series - lo) / (hi - lo)
    return series * 0.0


google = clip_window(
    get_google_mobility(iso3).rolling(smooth, center=True).mean()
)
fb = clip_window(
    pd.DataFrame(
        {
            "fb_visited_mob": get_fb_visited_mobility(iso3),
            "fb_singletile_mob": get_fb_singletile_mobility(iso3),
        }
    ).rolling(smooth, center=True).mean()
)
pol = 1.0 - clip_window(
    get_oxcgrt(iso3, "custom").rolling(smooth, center=True).mean()
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
linewidth = 2.0

g_cols = [c for c in google.columns if c != "residential"]
for col in g_cols:
    ax.plot(
        google.index,
        minmax_within_window(google[col]),
        color=google_colour,
        alpha=0.45,
        lw=linewidth,
    )

for col in fb.columns:
    ax.plot(
        fb.index,
        minmax_within_window(fb[col]),
        color=facebook_colour,
        alpha=0.85,
        lw=linewidth,
    )

for col in pol.columns:
    ax.plot(
        pol.index,
        minmax_within_window(pol[col]),
        color=oxcgrt_colour,
        alpha=0.4,
        lw=linewidth,
    )

ax.set_ylabel("relative level within window")
ax.legend(
    handles=[
        Line2D([0], [0], color=google_colour, lw=linewidth, label="Google mobility"),
        Line2D([0], [0], color=facebook_colour, lw=linewidth, label="Facebook mobility"),
        Line2D([0], [0], color=oxcgrt_colour, lw=linewidth, label="OxCGRT policies"),
    ],
    loc="upper right",
    framealpha=0.85,
)
fig.tight_layout()
fig.savefig("raw_drivers.pdf")

Note that residential is omitted from Google

Note that policies are the reciprocal